# SVA Sentinel: OpenVINO NNCF INT8 Quantization & Edge Artifact Export
## Post-Training Quantization of Distilled Student Detector (YOLO11n)

**Feature ID:** `SVA-QUANT-001`  
**Target Environment:** Google Colab Environment & Edge CPU Packaging  
**Objective:** Quantize the distilled student detector (`yolo11n_distilled.onnx`) to 8-bit integer precision (INT8) using OpenVINO NNCF with a 300-frame calibration dataset (day/dusk/night), achieving $\ge 60\%$ model size reduction and $\le 1.5\%$ mAP drop, and package OpenVINO IR (`.xml`/`.bin`) artifacts for real-time edge CPU inference ($\ge 25\text{ FPS}$, $< 1.2\text{GB}$ RAM).

### Acceptance Criteria Coverage:
- **R1-R3:** Colab environment configuration, Drive mounting, and OpenVINO/NNCF setup
- **R4:** Fail-fast verification on missing distilled input checkpoint
- **R5:** OpenVINO NNCF Post-Training Quantization (`nncf.quantize` INT8)
- **R6-R10:** 300-frame calibration set balanced across day (120), dusk (90), night (90) conditions
- **R11, R16:** Size reduction $\ge 60\%$ and OpenVINO IR export (`yolo11n_int8.xml`, `yolo11n_int8.bin`)
- **R12-R15:** Accuracy retention check ($\Delta mAP@50-95 \le 1.5\%$) across canonical 5 classes
- **R17-R19:** Laptop CPU benchmark execution ($\ge 25\text{ FPS}$, RSS $< 1.2\text{GB}$ RAM)

### Step 0: Clone Repository and Environment Setup
Clone the official `sva_sentinel` codebase and install dependencies in editable mode so all modules in `src/` are accessible globally.

In [ ]:
import os, sys
from pathlib import Path

# Clone repository if not already present
repo_dir = Path('/content/sva_sentinel')
if not repo_dir.exists():
    !git clone https://github.com/Many871027/sva_sentinel.git /content/sva_sentinel

# Add to sys.path and set working directory
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
%cd /content/sva_sentinel

# Install package in editable mode and install required dependencies
!pip install -q -e .
!pip install -q ultralytics onnx onnxruntime mlflow pydantic supervision psutil
print('[OK] SVA Sentinel codebase and dependencies installed successfully.')


### Step 1: Environment Setup & Optimization Dependencies (R1, R3)
Install OpenVINO runtime, NNCF compression framework, and system metrics utilities.

In [ ]:
!pip install -q -U openvino nncf opencv-python-headless psutil pydantic

import openvino as ov
import nncf
import psutil
print(f"[OK] OpenVINO version: {ov.__version__}")
print(f"[OK] NNCF version: {nncf.__version__}")

### Step 2: Google Drive Mounting & Input Verification (R2, R4)
Mount Google Drive and verify the presence of the distilled student detector.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = Path('/content/drive/MyDrive/sva_sentinel')
    print(f"[OK] Google Drive mounted at {drive_dir}")
except ImportError:
    drive_dir = Path('./workspace_drive')
    drive_dir.mkdir(parents=True, exist_ok=True)
    print("[LOCAL] Operating in local workspace mode.")

model_fp32_path = drive_dir / 'models/distilled/yolo11n_distilled.onnx'
if not model_fp32_path.exists():
    # Fallback to local repo distilled model path if testing locally
    model_fp32_path = Path('models/distilled/yolo11n_distilled.onnx')

print(f"Input model path: {model_fp32_path} (Exists: {model_fp32_path.exists()})")

### Step 3: Representative 300-Frame Calibration Dataset (R6-R10)
Assemble 300 unlabelled intersection frames balanced across day (120), dusk (90), and night (90).

In [ ]:
import sys
from pathlib import Path
possible_roots = [Path.cwd(), Path("/content/sva_sentinel"), Path("/content/ROADWAY-MONITOR"), Path.cwd() / "sva_sentinel"]
repo_root = next((p for p in possible_roots if (p / "src").is_dir()), Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.pipeline.calibration_loader import CalibrationFrameLoader

loader = CalibrationFrameLoader(target_frames=300, seed=42)
frames, manifest = loader.assemble_calibration_frames()

print(f"[OK] Assembled {manifest.total_frames} frames:")
print(f"  - Day (40%): {manifest.day_frames}")
print(f"  - Dusk (30%): {manifest.dusk_frames}")
print(f"  - Night (30%): {manifest.night_frames}")
print(f"Tensor shape: {manifest.input_resolution}")

### Step 4: OpenVINO NNCF Post-Training Quantization (PTQ) (R5, R11, R16)
Run `nncf.quantize` with `PERFORMANCE` preset and export OpenVINO IR (`.xml` and `.bin`).

In [ ]:
from src.engine.openvino_quantizer import OpenVINOQuantizer, QuantizationConfig

export_dir = drive_dir / 'models/quantized/openvino'
cfg = QuantizationConfig(
    model_input_path=str(model_fp32_path),
    output_dir=str(export_dir),
    preset='PERFORMANCE',
    fast_bias_correction=True,
)

quantizer = OpenVINOQuantizer(config=cfg)
report = quantizer.quantize(calibration_frames=frames)

print("=== Quantization Summary Report ===")
print(f"FP32 Model Size: {report.baseline_fp32_size_bytes / (1024*1024):.2f} MB")
print(f"INT8 Model Size: {report.quantized_int8_size_bytes / (1024*1024):.2f} MB")
print(f"Size Reduction: {report.size_reduction_pct:.1f}% (Target >= 60%): {report.achieved_size_reduction}")
print(f"mAP@50-95 Drop: {report.map_drop:.4f} (Target <= 0.015): {report.achieved_accuracy_tolerance}")
print(f"Exported XML: {report.ir_xml_path}")
print(f"Exported BIN: {report.ir_bin_path}")

### Step 5: Edge CPU Latency and Memory Benchmark (R17-R19)
Benchmark 100 iterations on CPU, verifying $\ge 25\text{ FPS}$ throughput and peak RSS $< 1.2\text{GB}$.

In [ ]:
from src.engine.openvino_benchmark import OpenVINOBenchmarkRunner

runner = OpenVINOBenchmarkRunner(device_name='CPU', benchmark_iterations=100)
bench_res = runner.benchmark_model(xml_path=report.ir_xml_path)

print("=== Edge CPU Benchmark Results ===")
print(f"Throughput: {bench_res['throughput_fps']} FPS (Target >= 25 FPS): {bench_res['achieved_target_fps']}")
print(f"Mean Latency: {bench_res['mean_latency_ms']} ms")
print(f"P95 Latency: {bench_res['p95_latency_ms']} ms")
print(f"Peak RSS Memory: {bench_res['peak_rss_mb']:.1f} MB (Target < 1200MB): {bench_res['within_memory_budget']}")